# PySpark Functions — Student Practice Lab
### Fill-in-the-blank & Write-from-scratch Exercises

---

> **How to use this notebook:**
> - Read each instruction carefully before writing any code
> - For **fill-in-the-blank** exercises, replace every `___` with the correct value
> - For **write-from-scratch** exercises, write your full solution where you see `# YOUR CODE HERE`
> - Run each cell to check your answer — errors mean something needs fixing
> - The **Cheat Sheet** in Section 1 is always available to reference
> - Do NOT look at the answer key until you have genuinely tried each exercise

---

**Scoring guide (track your own progress):**
| Section | Exercises | Points each | Total |
|---------|-----------|-------------|-------|
| 2 — DataFrame Operations | 10 | 2 | 20 |
| 3 — Built-in & Aggregates | 8  | 2 | 16 |
| 4 — String Functions       | 8  | 2 | 16 |
| 5 — Date & Time Functions  | 6  | 3 | 18 |
| 6 — Array Functions        | 4  | 3 | 12 |
| 7 — Capstone Challenge     | 3  | 6 | 18 |
| **Total** | | | **100** |


## Section 0 — Colab Setup (Run First)

> Run this cell once before starting any exercises.


In [4]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq
!pip install -q pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

In [6]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('PySpark Student Lab') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('PySpark Student Lab') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'Ready! Spark {spark.version}')

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, MapType
from pyspark.sql.window import Window

# Create a SparkSession – the entry point for all PySpark operations
# .master("local[*]") runs locally using all available CPU cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PySpark 100 Functions Lab") \
    .getOrCreate()

# Suppress verbose INFO logs for cleaner output
spark.sparkContext.setLogLevel("ERROR")

print("✅ SparkSession created successfully!")
print(f"   Spark version: {spark.version}")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [9]:
!apt-get update -qq
!pip install -q pyspark
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [10]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PySpark Student Lab") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Ready! Spark {spark.version}")

Ready! Spark 4.0.2


---
## Section 1 — Cheat Sheet (Keep this open while working)

### How PySpark Works — Key Mental Model

```
YOUR CODE (transformations — lazy, nothing runs yet)
   df.filter(...).groupBy(...).agg(...)
              |
              v
   CATALYST OPTIMISER builds a plan
              |
              v
   ACTION triggered (show / count / write)
              |
              v
   DAG -> Stages -> Tasks -> Partitions processed in parallel
```

---

### 1A — DataFrame Operations Quick Reference

| Function | Syntax | What it does |
|----------|--------|-------------|
| `show()` | `df.show(n)` | Display first n rows (default 20) |
| `printSchema()` | `df.printSchema()` | Show column names and types |
| `select()` | `df.select("col1", "col2")` | Keep only specified columns |
| `filter()` / `where()` | `df.filter(F.col("x") > 5)` | Keep rows matching condition |
| `withColumn()` | `df.withColumn("new", expr)` | Add or replace a column |
| `withColumnRenamed()` | `df.withColumnRenamed("old","new")` | Rename a column |
| `drop()` | `df.drop("col")` | Remove a column |
| `dropDuplicates()` | `df.dropDuplicates(["col"])` | Remove duplicate rows |
| `orderBy()` | `df.orderBy("col", ascending=False)` | Sort rows |
| `groupBy().agg()` | `df.groupBy("col").agg(F.sum("x"))` | Group and aggregate |
| `join()` | `df.join(df2, on="key", how="inner")` | Combine two DataFrames |
| `union()` | `df.union(df2)` | Stack DataFrames vertically |
| `fillna()` | `df.fillna({"col": value})` | Fill null values |
| `limit()` | `df.limit(n)` | Return first n rows |
| `collect()` | `df.collect()` | Return all rows as Python list |
| `count()` | `df.count()` | Count total rows |

```
HOW select() WORKS:
  df with cols [id, name, dept, salary]
      |
      v  df.select("id", "name")
      |
      v  result: [id, name]   <-- only selected columns remain
```

```
HOW filter() WORKS:
  df rows: [1,Alice] [2,Bob] [3,Clara]
      |
      v  df.filter(F.col("id") > 1)
      |
      v  result: [2,Bob] [3,Clara]  <-- rows not matching are dropped
```

```
HOW groupBy().agg() WORKS:
  rows: Eng,90k | Mkt,70k | Eng,80k | HR,60k
      |
      v  .groupBy("dept").agg(F.avg("salary"))
      |
      v  Eng: avg(90k,80k)=85k
         Mkt: avg(70k)=70k         <-- one row per group
         HR:  avg(60k)=60k
```

```
HOW join() WORKS:
  left:  [1,Alice] [2,Bob] [3,Clara]
  right: [1,Math]  [2,Science]
      |
      v  left.join(right, on="id", how="inner")
      |
      v  [1,Alice,Math] [2,Bob,Science]  <-- only matching ids kept
```

---

### 1B — Built-in Column & Aggregate Functions

| Function | Syntax | What it does |
|----------|--------|-------------|
| `F.col()` | `F.col("name")` | Reference a column |
| `F.lit()` | `F.lit(42)` | Create a literal constant column |
| `F.when()` | `F.when(cond, val).otherwise(val)` | Conditional (CASE WHEN) |
| `F.isNull()` | `F.col("x").isNull()` | Check for NULL |
| `F.isNotNull()` | `F.col("x").isNotNull()` | Check for NOT NULL |
| `F.count()` | `F.count("col")` | Count non-null values |
| `F.sum()` | `F.sum("col")` | Sum of column |
| `F.avg()` | `F.avg("col")` | Average of column |
| `F.min()` | `F.min("col")` | Minimum value |
| `F.max()` | `F.max("col")` | Maximum value |
| `F.countDistinct()` | `F.countDistinct("col")` | Count unique values |
| `F.stddev()` | `F.stddev("col")` | Standard deviation |
| `F.alias()` | `F.sum("x").alias("total")` | Rename output column |
| `F.cast()` | `F.col("x").cast("int")` | Change data type |
| `F.expr()` | `F.expr("salary * 0.1")` | SQL expression as string |

```
HOW when().otherwise() WORKS:
  F.when(condition_1, value_1)
   .when(condition_2, value_2)
   .otherwise(default_value)

  Example:
  F.when(F.col("salary") >= 90000, "Senior")
   .when(F.col("salary") >= 70000, "Mid")
   .otherwise("Junior")

  salary=95000 -> "Senior"
  salary=75000 -> "Mid"
  salary=50000 -> "Junior"
```

---

### 1C — Window Functions

| Function | What it does |
|----------|-------------|
| `F.rank().over(w)` | Rank with gaps (1,1,3) |
| `F.dense_rank().over(w)` | Rank without gaps (1,1,2) |
| `F.row_number().over(w)` | Unique sequential number |
| `F.lead("col", n).over(w)` | Value from n rows AHEAD |
| `F.lag("col", n).over(w)` | Value from n rows BEHIND |
| `F.sum("col").over(w)` | Running total |

```
HOW Window functions WORK:
  windowSpec = Window.partitionBy("dept").orderBy("salary")

  WITHOUT groupBy —> every row keeps its output:
  id | dept | salary | rank_in_dept
   1 |  Eng  | 80000  |      1
   2 |  Eng  | 90000  |      2       <- ranked WITHIN dept
   3 |  Mkt  | 70000  |      1
   4 |  Mkt  | 75000  |      2       <- separate rank for Mkt
```

---

### 1D — String Functions

| Function | Syntax | What it does |
|----------|--------|-------------|
| `F.upper()` | `F.upper("col")` | Convert to UPPERCASE |
| `F.lower()` | `F.lower("col")` | Convert to lowercase |
| `F.length()` | `F.length("col")` | Number of characters |
| `F.trim()` | `F.trim("col")` | Remove leading/trailing spaces |
| `F.concat()` | `F.concat("col1", F.lit("-"), "col2")` | Join strings together |
| `F.concat_ws()` | `F.concat_ws(",", "col1", "col2")` | Join with separator |
| `F.substring()` | `F.substring("col", start, length)` | Extract part of string (1-indexed) |
| `F.split()` | `F.split("col", delimiter)` | Split string into array |
| `F.regexp_replace()` | `F.regexp_replace("col", pattern, replacement)` | Replace via regex |
| `F.regexp_extract()` | `F.regexp_extract("col", pattern, group)` | Extract via regex |
| `F.initcap()` | `F.initcap("col")` | Capitalise First Letter Of Each Word |
| `F.lpad()` | `F.lpad("col", length, pad_char)` | Pad string on the LEFT |
| `F.rpad()` | `F.rpad("col", length, pad_char)` | Pad string on the RIGHT |
| `F.reverse()` | `F.reverse("col")` | Reverse the string |
| `F.instr()` | `F.instr("col", "substring")` | Position of first occurrence (0 = not found) |
| `F.translate()` | `F.translate("col", "from_chars", "to_chars")` | Map characters one-to-one |

```
HOW substring() WORKS (1-indexed!):
  F.substring("name", 1, 3)
    position 1 2 3 4 5
    "Alice"  A l i c e
              ^^^
    result: "Ali"
```

```
HOW split() WORKS:
  F.split("name", "l")
  "Alice" split on "l" -> ["A", "ice"]
  "Bob"   split on "l" -> ["Bo", ""]    (l is last char)
  "Charlie" split on "l" -> ["Char", "ie"]
```

---

### 1E — Date & Time Functions

| Function | Syntax | What it does |
|----------|--------|-------------|
| `F.current_date()` | `F.current_date()` | Today's date |
| `F.current_timestamp()` | `F.current_timestamp()` | Current date + time |
| `F.to_date()` | `F.to_date("col", "yyyy-MM-dd")` | Parse string to date |
| `F.to_timestamp()` | `F.to_timestamp("col", "format")` | Parse string to timestamp |
| `F.date_add()` | `F.date_add("col", n)` | Add n days to date |
| `F.date_sub()` | `F.date_sub("col", n)` | Subtract n days from date |
| `F.datediff()` | `F.datediff("end", "start")` | Days between two dates |
| `F.year()` | `F.year("col")` | Extract year |
| `F.month()` | `F.month("col")` | Extract month (1-12) |
| `F.dayofmonth()` | `F.dayofmonth("col")` | Extract day of month |
| `F.dayofweek()` | `F.dayofweek("col")` | Day of week (1=Sun, 7=Sat) |
| `F.hour()` | `F.hour("col")` | Extract hour |
| `F.date_format()` | `F.date_format("col", "dd-MMM-yyyy")` | Format date as string |
| `F.add_months()` | `F.add_months("col", n)` | Add n months |
| `F.last_day()` | `F.last_day("col")` | Last day of the month |
| `F.trunc()` | `F.trunc("col", "MM")` | Truncate to start of month/year |

```
HOW datediff() WORKS:
  F.datediff(end_date, start_date)
  datediff("2025-06-01", "2025-01-01") = 151 days
  (positive = end is after start)
```

---

### 1F — Array Functions

| Function | Syntax | What it does |
|----------|--------|-------------|
| `F.array()` | `F.array("col1", "col2")` | Create an array from columns |
| `F.array_contains()` | `F.array_contains("col", value)` | True if value is in array |
| `F.array_distinct()` | `F.array_distinct("col")` | Remove duplicates from array |
| `F.array_intersect()` | `F.array_intersect("col1", "col2")` | Common elements of two arrays |
| `F.array_union()` | `F.array_union("col1", "col2")` | All elements from both arrays (deduplicated) |
| `F.explode()` | `F.explode("col")` | One row per array element |
| `F.size()` | `F.size("col")` | Number of elements in array |

```
HOW explode() WORKS:
  BEFORE:
  id | tags
   1 | ["python","spark","sql"]
   2 | ["java","scala"]

  AFTER explode("tags"):
  id | tag
   1 | python
   1 | spark        <- one row per element
   1 | sql
   2 | java
   2 | scala
```


---
## Section 2 — Dataset Setup

Run this cell first. All exercises use these DataFrames.


In [12]:
# === BASE DATAFRAME — used in most exercises ===
df = spark.createDataFrame([
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Marketing",   72000),
    (3, "Clara",   "Engineering", 88000),
    (4, "Dan",     "HR",          65000),
    (5, "Eva",     "Marketing",   79000),
    (6, "Frank",   "Engineering", 91000),
    (7, "Grace",   "HR",          70000),
    (8, "Henry",   "Marketing",   68000),
], ["id", "name", "dept", "salary"])

# === DATE DATAFRAME — used in date exercises ===
df_dates = spark.createDataFrame([
    (1, "Alice", "2023-03-15"),
    (2, "Bob",   "2022-11-01"),
    (3, "Clara", "2024-01-20"),
    (4, "Dan",   "2021-07-04"),
], ["id", "name", "hire_date"])

# === PRODUCTS DATAFRAME — used in array exercises ===
df_products = spark.createDataFrame([
    (1, "Laptop",  ["electronics", "computers", "work"]),
    (2, "T-Shirt", ["clothing", "fashion"]),
    (3, "Phone",   ["electronics", "mobile", "electronics"]),
    (4, "Headset", ["electronics", "audio", "computers"]),
], ["id", "product", "tags"])

print("All DataFrames ready!")
df.show()

All DataFrames ready!
+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1|Alice|Engineering| 95000|
|  2|  Bob|  Marketing| 72000|
|  3|Clara|Engineering| 88000|
|  4|  Dan|         HR| 65000|
|  5|  Eva|  Marketing| 79000|
|  6|Frank|Engineering| 91000|
|  7|Grace|         HR| 70000|
|  8|Henry|  Marketing| 68000|
+---+-----+-----------+------+



---
## Section 3 — DataFrame Operations

---
### Exercise 3.1 — `select()` [Fill-in-the-blank]

**Task:** Select only the `name` and `salary` columns from `df`.

**Expected output:**
```
+-------+------+
|   name|salary|
+-------+------+
|  Alice| 95000|
|    Bob| 72000|
...
```


In [13]:
# Fill in the blank: replace ___ with the correct column names
result = df.select('name', 'salary')
result.show()

+-----+------+
| name|salary|
+-----+------+
|Alice| 95000|
|  Bob| 72000|
|Clara| 88000|
|  Dan| 65000|
|  Eva| 79000|
|Frank| 91000|
|Grace| 70000|
|Henry| 68000|
+-----+------+



---
### Exercise 3.2 — `filter()` [Fill-in-the-blank]

**Task:** Keep only employees with salary **greater than** 75000.


In [14]:
# Fill in the blank: replace ___ with the correct operator and value
result = df.filter(F.col('salary') > 75000)
result.show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1|Alice|Engineering| 95000|
|  3|Clara|Engineering| 88000|
|  5|  Eva|  Marketing| 79000|
|  6|Frank|Engineering| 91000|
+---+-----+-----------+------+



---
### Exercise 3.3 — `withColumn()` [Fill-in-the-blank]

**Task:** Add a new column called `bonus` equal to 10% of salary (`salary * 0.1`).


In [15]:
# Fill in the blank
result = df.withColumn('bonus', F.col('salary') * 0.1)
result.show()

+---+-----+-----------+------+------+
| id| name|       dept|salary| bonus|
+---+-----+-----------+------+------+
|  1|Alice|Engineering| 95000|9500.0|
|  2|  Bob|  Marketing| 72000|7200.0|
|  3|Clara|Engineering| 88000|8800.0|
|  4|  Dan|         HR| 65000|6500.0|
|  5|  Eva|  Marketing| 79000|7900.0|
|  6|Frank|Engineering| 91000|9100.0|
|  7|Grace|         HR| 70000|7000.0|
|  8|Henry|  Marketing| 68000|6800.0|
+---+-----+-----------+------+------+



---
### Exercise 3.4 — `withColumnRenamed()` [Fill-in-the-blank]

**Task:** Rename the column `dept` to `department`.


In [16]:
# Fill in the blank
result = df.withColumnRenamed("dept", 'department')
result.show()

+---+-----+-----------+------+
| id| name| department|salary|
+---+-----+-----------+------+
|  1|Alice|Engineering| 95000|
|  2|  Bob|  Marketing| 72000|
|  3|Clara|Engineering| 88000|
|  4|  Dan|         HR| 65000|
|  5|  Eva|  Marketing| 79000|
|  6|Frank|Engineering| 91000|
|  7|Grace|         HR| 70000|
|  8|Henry|  Marketing| 68000|
+---+-----+-----------+------+



---
### Exercise 3.5 — `drop()` [Fill-in-the-blank]

**Task:** Remove the `id` column from `df`.


In [17]:
# Fill in the blank
result = df.drop('id')
result.show()

+-----+-----------+------+
| name|       dept|salary|
+-----+-----------+------+
|Alice|Engineering| 95000|
|  Bob|  Marketing| 72000|
|Clara|Engineering| 88000|
|  Dan|         HR| 65000|
|  Eva|  Marketing| 79000|
|Frank|Engineering| 91000|
|Grace|         HR| 70000|
|Henry|  Marketing| 68000|
+-----+-----------+------+



---
### Exercise 3.6 — `orderBy()` [Fill-in-the-blank]

**Task:** Sort `df` by salary in **descending** order.


In [21]:
# Fill in the blank — hint: ascending=False for descending
result = df.orderBy('salary', ascending=False)
result.show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1|Alice|Engineering| 95000|
|  6|Frank|Engineering| 91000|
|  3|Clara|Engineering| 88000|
|  5|  Eva|  Marketing| 79000|
|  2|  Bob|  Marketing| 72000|
|  7|Grace|         HR| 70000|
|  8|Henry|  Marketing| 68000|
|  4|  Dan|         HR| 65000|
+---+-----+-----------+------+



---
### Exercise 3.7 — `groupBy().agg()` [Fill-in-the-blank]

**Task:** Find the **average salary** per department. Name the output column `avg_salary`.


In [24]:
# Fill in the blank
result = df.groupBy('dept').agg(
    F.avg('salary').alias('avg_salary')
)
result.show()

+-----------+-----------------+
|       dept|       avg_salary|
+-----------+-----------------+
|Engineering|91333.33333333333|
|         HR|          67500.0|
|  Marketing|          73000.0|
+-----------+-----------------+



---
### Exercise 3.8 — `join()` [Fill-in-the-blank]

**Task:** Inner join `df` with `df2` below on the `id` column.


In [25]:
df2 = spark.createDataFrame([
    (1, "Python"),
    (2, "Java"),
    (3, "Scala"),
    (4, "R"),
], ["id", "language"])

# Fill in the blank
result = df.join(df2, on='id', how='inner')
result.show()

+---+-----+-----------+------+--------+
| id| name|       dept|salary|language|
+---+-----+-----------+------+--------+
|  1|Alice|Engineering| 95000|  Python|
|  2|  Bob|  Marketing| 72000|    Java|
|  3|Clara|Engineering| 88000|   Scala|
|  4|  Dan|         HR| 65000|       R|
+---+-----+-----------+------+--------+



---
### Exercise 3.9 — `when().otherwise()` [Fill-in-the-blank]

**Task:** Add a column `grade`:
- `"High"` if salary >= 90000
- `"Mid"` if salary >= 75000
- `"Standard"` otherwise


In [26]:
# Fill in the blank
result = df.withColumn("grade",
    F.when(F.col("salary") >= 90000, 'High')     .when(F.col("salary") >= 75000, 'Mid')     .otherwise('Standard')
)
result.show()

+---+-----+-----------+------+--------+
| id| name|       dept|salary|   grade|
+---+-----+-----------+------+--------+
|  1|Alice|Engineering| 95000|    High|
|  2|  Bob|  Marketing| 72000|Standard|
|  3|Clara|Engineering| 88000|     Mid|
|  4|  Dan|         HR| 65000|Standard|
|  5|  Eva|  Marketing| 79000|     Mid|
|  6|Frank|Engineering| 91000|    High|
|  7|Grace|         HR| 70000|Standard|
|  8|Henry|  Marketing| 68000|Standard|
+---+-----+-----------+------+--------+



---
### Exercise 3.10 — Window Function [Write from scratch]

**Task:** Using a window function, add a column `rank_in_dept` that ranks employees **within each department** by salary (highest salary = rank 1).

**Hints:**
- Use `Window.partitionBy("dept").orderBy(F.col("salary").desc())`
- Use `F.rank().over(windowSpec)`


In [28]:
# YOUR CODE HERE
# Step 1: define the window specification
windowSpec = Window.partitionBy("dept").orderBy(F.col("salary").desc())

# Step 2: add the rank column
result = df.withColumn("rank_in_dept", F.rank().over(windowSpec))

result.show()

+---+-----+-----------+------+------------+
| id| name|       dept|salary|rank_in_dept|
+---+-----+-----------+------+------------+
|  1|Alice|Engineering| 95000|           1|
|  6|Frank|Engineering| 91000|           2|
|  3|Clara|Engineering| 88000|           3|
|  7|Grace|         HR| 70000|           1|
|  4|  Dan|         HR| 65000|           2|
|  5|  Eva|  Marketing| 79000|           1|
|  2|  Bob|  Marketing| 72000|           2|
|  8|Henry|  Marketing| 68000|           3|
+---+-----+-----------+------+------------+



---
## Section 4 — Built-in & Aggregate Functions

---
### Exercise 4.1 — `F.lit()` [Fill-in-the-blank]

**Task:** Add a constant column called `company` with the value `"Acme Corp"` to `df`.


In [29]:
# Fill in the blank
result = df.withColumn("company", F.lit('Acme Corp'))
result.show()

+---+-----+-----------+------+---------+
| id| name|       dept|salary|  company|
+---+-----+-----------+------+---------+
|  1|Alice|Engineering| 95000|Acme Corp|
|  2|  Bob|  Marketing| 72000|Acme Corp|
|  3|Clara|Engineering| 88000|Acme Corp|
|  4|  Dan|         HR| 65000|Acme Corp|
|  5|  Eva|  Marketing| 79000|Acme Corp|
|  6|Frank|Engineering| 91000|Acme Corp|
|  7|Grace|         HR| 70000|Acme Corp|
|  8|Henry|  Marketing| 68000|Acme Corp|
+---+-----+-----------+------+---------+



---
### Exercise 4.2 — `F.cast()` [Fill-in-the-blank]

**Task:** Add a column `salary_str` which is the `salary` column converted to a **string** type.


In [30]:
# Fill in the blank
result = df.withColumn("salary_str", F.col('salary').cast('string'))
result.printSchema()
result.show()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- salary_str: string (nullable = true)

+---+-----+-----------+------+----------+
| id| name|       dept|salary|salary_str|
+---+-----+-----------+------+----------+
|  1|Alice|Engineering| 95000|     95000|
|  2|  Bob|  Marketing| 72000|     72000|
|  3|Clara|Engineering| 88000|     88000|
|  4|  Dan|         HR| 65000|     65000|
|  5|  Eva|  Marketing| 79000|     79000|
|  6|Frank|Engineering| 91000|     91000|
|  7|Grace|         HR| 70000|     70000|
|  8|Henry|  Marketing| 68000|     68000|
+---+-----+-----------+------+----------+



---
### Exercise 4.3 — `F.isNull()` / `F.isNotNull()` [Fill-in-the-blank]

**Task:** From `df_nulls` below, keep only rows where `email` is **not null**.


In [31]:
df_nulls = spark.createDataFrame([
    (1, "Alice", "alice@mail.com"),
    (2, "Bob",    None),
    (3, "Clara", "clara@mail.com"),
    (4, "Dan",    None),
], ["id", "name", "email"])

# Fill in the blank
result = df_nulls.filter(F.col("email").isNotNull())
result.show()

+---+-----+--------------+
| id| name|         email|
+---+-----+--------------+
|  1|Alice|alice@mail.com|
|  3|Clara|clara@mail.com|
+---+-----+--------------+



---
### Exercise 4.4 — `F.countDistinct()` [Fill-in-the-blank]

**Task:** Count how many **distinct** departments exist in `df`.


In [35]:
# Fill in the blank
result = df.select(F.countDistinct("dept").alias("unique_departments"))
result.show()

+------------------+
|unique_departments|
+------------------+
|                 3|
+------------------+



---
### Exercise 4.5 — Multiple Aggregates [Fill-in-the-blank]

**Task:** In a single `select()`, compute:
- Total salary (`sum`), aliased as `total_salary`
- Average salary (`avg`), aliased as `avg_salary`
- Minimum salary (`min`), aliased as `min_salary`
- Maximum salary (`max`), aliased as `max_salary`


In [36]:
# Fill in the blank
result = df.select(
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary"),
)
result.show()

+------------+----------+----------+----------+
|total_salary|avg_salary|min_salary|max_salary|
+------------+----------+----------+----------+
|      628000|   78500.0|     65000|     95000|
+------------+----------+----------+----------+



---
### Exercise 4.6 — `F.expr()` [Fill-in-the-blank]

**Task:** Using `F.expr()`, add a column `tax` equal to `salary * 0.2` using a **SQL string expression**.


In [38]:
# Fill in the blank — write the expression as a SQL string inside F.expr()
result = df.withColumn("tax", F.expr('salary*0.2'))
result.show()

+---+-----+-----------+------+-------+
| id| name|       dept|salary|    tax|
+---+-----+-----------+------+-------+
|  1|Alice|Engineering| 95000|19000.0|
|  2|  Bob|  Marketing| 72000|14400.0|
|  3|Clara|Engineering| 88000|17600.0|
|  4|  Dan|         HR| 65000|13000.0|
|  5|  Eva|  Marketing| 79000|15800.0|
|  6|Frank|Engineering| 91000|18200.0|
|  7|Grace|         HR| 70000|14000.0|
|  8|Henry|  Marketing| 68000|13600.0|
+---+-----+-----------+------+-------+



---
### Exercise 4.7 — `collect_list()` [Fill-in-the-blank]

**Task:** Group by `dept` and collect all employee names into a list column called `employees`.


In [39]:
# Fill in the blank
result = df.groupBy("dept").agg(
    F.collect_list("name").alias("employees")
)
result.show(truncate=False)

+-----------+---------------------+
|dept       |employees            |
+-----------+---------------------+
|Engineering|[Alice, Clara, Frank]|
|HR         |[Dan, Grace]         |
|Marketing  |[Bob, Eva, Henry]    |
+-----------+---------------------+



---
### Exercise 4.8 — Window `lead()` and `lag()` [Write from scratch]

**Task:** Using a window ordered by `id`, add two columns:
- `next_salary`: the salary of the **next** employee (lead)
- `prev_salary`: the salary of the **previous** employee (lag)

Rows with no next/previous should show `null`.

**Hints:**
- `F.lead("salary", 1).over(windowSpec)`
- `F.lag("salary", 1).over(windowSpec)`


In [40]:
# YOUR CODE HERE
windowSpec = Window.orderBy("id")

result = df \
    .withColumn("next_salary", F.lead("salary", 1).over(windowSpec)) \
    .withColumn("prev_salary", F.lag("salary", 1).over(windowSpec))

result.show()

+---+-----+-----------+------+-----------+-----------+
| id| name|       dept|salary|next_salary|prev_salary|
+---+-----+-----------+------+-----------+-----------+
|  1|Alice|Engineering| 95000|      72000|       NULL|
|  2|  Bob|  Marketing| 72000|      88000|      95000|
|  3|Clara|Engineering| 88000|      65000|      72000|
|  4|  Dan|         HR| 65000|      79000|      88000|
|  5|  Eva|  Marketing| 79000|      91000|      65000|
|  6|Frank|Engineering| 91000|      70000|      79000|
|  7|Grace|         HR| 70000|      68000|      91000|
|  8|Henry|  Marketing| 68000|       NULL|      70000|
+---+-----+-----------+------+-----------+-----------+



---
## Section 5 — String Functions

---
### Exercise 5.1 — `upper()`, `lower()`, `length()` [Fill-in-the-blank]

**Task:** Add three columns to `df`:
- `name_upper`: name in UPPERCASE
- `name_lower`: name in lowercase
- `name_len`: number of characters in name


In [ ]:
# Fill in the blank
result = df \
    .withColumn("name_upper", F.upper("name")) \
    .withColumn("name_lower", F.lower("name")) \
    .withColumn("name_len",   F.length("name"))

result.select("name", "name_upper", "name_lower", "name_len").show()


---
### Exercise 5.2 — `concat_ws()` [Fill-in-the-blank]

**Task:** Create a column `full_label` combining `name` and `dept` with ` — ` as separator.

Expected: `Alice — Engineering`, `Bob — Marketing`, etc.


In [44]:
# Fill in the blank
result = df.withColumn("full_label", F.concat_ws(' — ', "name", 'dept'))
result.select("full_label").show()

+-------------------+
|         full_label|
+-------------------+
|Alice — Engineering|
|    Bob — Marketing|
|Clara — Engineering|
|           Dan — HR|
|    Eva — Marketing|
|Frank — Engineering|
|         Grace — HR|
|  Henry — Marketing|
+-------------------+



---
### Exercise 5.3 — `substring()` [Fill-in-the-blank]

**Task:** Extract the **first 3 characters** of each name into a column `name_short`.

Remember: `substring()` positions are **1-indexed**.


In [45]:
# Fill in the blank: substring(column, start_position, length)
result = df.withColumn("name_short", F.substring("name", 1, 3))
result.select("name", "name_short").show()

+-----+----------+
| name|name_short|
+-----+----------+
|Alice|       Ali|
|  Bob|       Bob|
|Clara|       Cla|
|  Dan|       Dan|
|  Eva|       Eva|
|Frank|       Fra|
|Grace|       Gra|
|Henry|       Hen|
+-----+----------+



---
### Exercise 5.4 — `regexp_replace()` [Fill-in-the-blank]

**Task:** In the `dept` column, replace the word `"Engineering"` with `"Eng"`.


In [48]:
# Fill in the blank
result = df.withColumn("dept_short", F.regexp_replace("dept", "Engineering", "Eng"))
result.select("dept", "dept_short").show()

+-----------+----------+
|       dept|dept_short|
+-----------+----------+
|Engineering|       Eng|
|  Marketing| Marketing|
|Engineering|       Eng|
|         HR|        HR|
|  Marketing| Marketing|
|Engineering|       Eng|
|         HR|        HR|
|  Marketing| Marketing|
+-----------+----------+



---
### Exercise 5.5 — `split()` [Fill-in-the-blank]

**Task:** Split the `name` column on the letter `"a"` into an array column called `name_parts`.


In [49]:
# Fill in the blank
result = df.withColumn("name_parts", F.split("name", 'a'))
result.select("name", "name_parts").show()

+-----+----------+
| name|name_parts|
+-----+----------+
|Alice|   [Alice]|
|  Bob|     [Bob]|
|Clara| [Cl, r, ]|
|  Dan|    [D, n]|
|  Eva|    [Ev, ]|
|Frank|  [Fr, nk]|
|Grace|  [Gr, ce]|
|Henry|   [Henry]|
+-----+----------+



---
### Exercise 5.6 — `lpad()` / `rpad()` [Fill-in-the-blank]

**Task:** Add two columns:
- `id_padded`: the `id` column (cast to string) padded with zeros on the **left** to a total length of 4. E.g. `"0001"`, `"0002"`.
- `name_padded`: the `name` column padded with `"."` on the **right** to a total length of 10.


In [51]:
# Fill in the blank
result = df \
    .withColumn("id_padded",   F.lpad(F.col("id").cast("string"), 4, '0')) \
    .withColumn("name_padded", F.rpad(F.col("name"), 10, '.'))

result.select("id", "id_padded", "name", "name_padded").show()

+---+---------+-----+-----------+
| id|id_padded| name|name_padded|
+---+---------+-----+-----------+
|  1|     0001|Alice| Alice.....|
|  2|     0002|  Bob| Bob.......|
|  3|     0003|Clara| Clara.....|
|  4|     0004|  Dan| Dan.......|
|  5|     0005|  Eva| Eva.......|
|  6|     0006|Frank| Frank.....|
|  7|     0007|Grace| Grace.....|
|  8|     0008|Henry| Henry.....|
+---+---------+-----+-----------+



---
### Exercise 5.7 — `initcap()` and `trim()` [Fill-in-the-blank]

**Task:** From `df_messy` below:
1. Trim whitespace from `name`
2. Apply `initcap()` to convert to proper case


In [52]:
df_messy = spark.createDataFrame([
    (1, "  ALICE SMITH  "),
    (2, "bob jones"),
    (3, " CLARA CHEN "),
], ["id", "name"])

# Fill in the blank — chain trim then initcap
result = df_messy \
    .withColumn("name", F.trim(F.col("name"))) \
    .withColumn("name", F.initcap(F.col("name")))

result.show()

+---+-----------+
| id|       name|
+---+-----------+
|  1|Alice Smith|
|  2|  Bob Jones|
|  3| Clara Chen|
+---+-----------+



---
### Exercise 5.8 — String Pipeline [Write from scratch]

**Task:** Given `df_emails` below, write a complete pipeline that:
1. Trims whitespace from `email`
2. Converts `email` to lowercase
3. Extracts the **username** (the part before `@`) into a new column `username`
4. Extracts the **domain** (the part after `@`) into a new column `domain`

**Hint:** Use `F.regexp_extract(col, pattern, group)` — group 1 captures what is inside the first `()`.
- Username pattern: `r'^(\S+)@'`
- Domain pattern: `r'@([\w.]+)$'`


In [ ]:
df_emails = spark.createDataFrame([
    (1, "  Alice@Gmail.COM  "),
    (2, "BOB@Company.org"),
    (3, "  clara@Yahoo.com  "),
], ["id", "email"])

# YOUR CODE HERE
result = df_emails \
    .withColumn("email", F.trim(F.col("email"))) \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("username", F.regexp_extract("email", r'^(\S+)@', 1)) \
    .withColumn("domain",   F.regexp_extract("email", r'@([\w.]+)$', 1))

result.show(truncate=False)


---
## Section 6 — Date & Time Functions

---
### Exercise 6.1 — `to_date()` and `datediff()` [Fill-in-the-blank]

**Task:** Using `df_dates`:
1. Convert `hire_date` string to a proper `DateType` column (format is `yyyy-MM-dd`)
2. Add column `days_employed` = number of days from `hire_date` to today


In [ ]:
# Fill in the blank
result = df_dates \
    .withColumn("hire_date",    F.to_date("hire_date", "yyyy-MM-dd")) \
    .withColumn("days_employed", F.datediff(F.current_date(), "hire_date"))

result.show()


---
### Exercise 6.2 — `date_add()` and `add_months()` [Fill-in-the-blank]

**Task:** Add two columns to `df_dates` (after converting `hire_date` to a date):
- `trial_end`: hire_date + **90 days**
- `first_review`: hire_date + **6 months**


In [ ]:
df_d = df_dates.withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))

# Fill in the blank
result = df_d \
    .withColumn("trial_end",    F.date_add("hire_date", 90)) \
    .withColumn("first_review", F.add_months("hire_date", 6))

result.show()


---
### Exercise 6.3 — Date Part Extraction [Fill-in-the-blank]

**Task:** Extract year, month, day of week, and week of year from `hire_date` into separate columns.

**Reminder:** `dayofweek()` returns 1=Sunday, 7=Saturday.


In [ ]:
df_d = df_dates.withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))

# Fill in the blank
result = df_d.select(
    "name",
    "hire_date",
    F.year("hire_date").alias("year"),
    F.month("hire_date").alias("month"),
    F.dayofweek("hire_date").alias("day_of_week"),
    F.weekofyear("hire_date").alias("week_of_year"),
)
result.show()


---
### Exercise 6.4 — `date_format()` and `last_day()` [Fill-in-the-blank]

**Task:** Add two columns:
- `hire_formatted`: format `hire_date` as `dd-MMM-yyyy` (e.g. `15-Mar-2023`)
- `month_end`: the last day of the month in which the employee was hired


In [ ]:
df_d = df_dates.withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))

# Fill in the blank
result = df_d \
    .withColumn("hire_formatted", F.date_format("hire_date", "dd-MMM-yyyy")) \
    .withColumn("month_end",      F.last_day("hire_date"))

result.show()


---
### Exercise 6.5 — `trunc()` [Fill-in-the-blank]

**Task:** Truncate `hire_date` to the **start of the month** and also to the **start of the year**.

**Hint:** Format strings are `"MM"` for month and `"YYYY"` for year.


In [ ]:
df_d = df_dates.withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))

# Fill in the blank
result = df_d \
    .withColumn("month_start", F.trunc("hire_date", "MM")) \
    .withColumn("year_start",  F.trunc("hire_date", "YYYY"))

result.show()


---
### Exercise 6.6 — Date Pipeline [Write from scratch]

**Task:** Build a complete date analysis pipeline on `df_dates` that:
1. Converts `hire_date` to a proper `DateType`
2. Calculates `days_employed` (days from hire to today)
3. Adds column `tenure_group`:
   - `"Veteran"` if days_employed > 1000
   - `"Experienced"` if days_employed > 500
   - `"New"` otherwise
4. Adds column `next_anniversary` = hire_date + 365 days
5. Shows only: `name`, `hire_date`, `days_employed`, `tenure_group`, `next_anniversary`


In [ ]:
# YOUR CODE HERE
result = df_dates \
    .withColumn("hire_date",     F.to_date("hire_date", "yyyy-MM-dd")) \
    .withColumn("days_employed", F.datediff(F.current_date(), "hire_date")) \
    .withColumn("tenure_group",
        F.when(F.col("days_employed") > 1000, "Veteran")
         .when(F.col("days_employed") > 500,  "Experienced")
         .otherwise("New")) \
    .withColumn("next_anniversary", F.date_add("hire_date", 365)) \
    .select("name", "hire_date", "days_employed", "tenure_group", "next_anniversary")

result.show()


---
## Section 7 — Array Functions

---
### Exercise 7.1 — `array_contains()` and `size()` [Fill-in-the-blank]

**Task:** Using `df_products`:
1. Filter to only products that contain the tag `"electronics"`
2. Add a column `tag_count` with the number of tags per product


In [ ]:
# Fill in the blank
result = df_products \
    .filter(F.array_contains("tags", "electronics")) \
    .withColumn("tag_count", F.size("tags"))

result.show(truncate=False)


---
### Exercise 7.2 — `array_distinct()` [Fill-in-the-blank]

**Task:** The `tags` column in `df_products` has duplicate entries (e.g. Phone has `"electronics"` twice). Add a column `unique_tags` with duplicates removed.


In [ ]:
# Fill in the blank
result = df_products.withColumn("unique_tags", F.array_distinct("tags"))
result.select("product", "tags", "unique_tags").show(truncate=False)


---
### Exercise 7.3 — `explode()` and `groupBy()` [Fill-in-the-blank]

**Task:** Use `F.explode()` to create one row per tag, then count how many products each tag appears in. Sort by count descending.


In [ ]:
# Fill in the blank
result = df_products \
    .withColumn("tag", F.explode("tags")) \
    .groupBy("tag") \
    .agg(F.count("id").alias("product_count")) \
    .orderBy("product_count", ascending=False)

result.show()


---
### Exercise 7.4 — `array_union()` and `array_intersect()` [Write from scratch]

**Task:** Join `df_products` with `df_promo` below (on `id`), then:
1. Add column `all_tags` = union of `tags` and `promo_tags` (no duplicates)
2. Add column `common_tags` = intersection of `tags` and `promo_tags`
3. Show `product`, `all_tags`, `common_tags`


In [ ]:
df_promo = spark.createDataFrame([
    (1, ["sale", "featured"]),
    (2, ["new",  "sale"]),
    (3, ["featured"]),
    (4, ["sale", "electronics"]),
], ["id", "promo_tags"])

# YOUR CODE HERE
result = df_products \
    .join(df_promo, on="id", how="inner") \
    .withColumn("all_tags",    F.array_union("tags", "promo_tags")) \
    .withColumn("common_tags", F.array_intersect("tags", "promo_tags")) \
    .select("product", "all_tags", "common_tags")

result.show(truncate=False)


---
## Section 8 — Capstone Challenge

This is your final challenge. No blanks — write everything from scratch.

### Dataset


In [ ]:
# Run this to set up the capstone dataset
orders = spark.createDataFrame([
    (1,  "alice smith",  "Electronics",  1200.50, "2024-03-15", ["laptop","sale","premium"]),
    (2,  "BOB JONES  ",  "Clothing",      45.00,  "2023-11-01", ["shirt","sale"]),
    (3,  "  Alice Smith", "Electronics",  89.99,  "2024-04-20", ["phone","electronics","electronics"]),
    (4,  "charlie brown", "Home",         210.00, "2024-05-25", ["decor","sale","decor"]),
    (5,  "BOB JONES",    "Clothing",       30.00, "2022-06-15", ["pants"]),
    (6,  "diana prince", "Electronics",  500.00,  "2024-05-28", ["laptop","premium","premium"]),
    (7,  "ALICE SMITH",  "Home",           75.00, "2023-08-10", ["lamp","decor"]),
    (8,  "eve wilson",   "Electronics",  320.00,  "2024-01-05", ["tablet","sale"]),
], ["order_id", "customer_name", "category", "amount", "order_date", "tags"])
print("Capstone dataset ready!")
orders.show(truncate=False)

### Capstone Task 1 — Data Cleaning & Enrichment [Write from scratch]

Build a pipeline that produces a clean, enriched `orders_clean` DataFrame:

1. **Clean `customer_name`**: trim whitespace and convert to proper case (e.g. `"Alice Smith"`)
2. **Convert `order_date`** string to a proper `DateType` (format: `yyyy-MM-dd`)
3. **Add `days_since_order`**: number of days between `order_date` and today
4. **Add `order_age`** column:
   - `"Recent"` if days_since_order <= 90
   - `"This Year"` if days_since_order <= 365
   - `"Old"` otherwise
5. **Add `unique_tags`**: deduplicated version of the `tags` array
6. **Add `tag_count`**: number of unique tags per order

Show the final result.


In [ ]:
# YOUR CODE HERE — Capstone Task 1
orders_clean = orders \
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name")))) \
    .withColumn("order_date",    F.to_date("order_date", "yyyy-MM-dd")) \
    .withColumn("days_since_order", F.datediff(F.current_date(), "order_date")) \
    .withColumn("order_age",
        F.when(F.col("days_since_order") <= 90,  "Recent")
         .when(F.col("days_since_order") <= 365, "This Year")
         .otherwise("Old")) \
    .withColumn("unique_tags", F.array_distinct("tags")) \
    .withColumn("tag_count",   F.size("unique_tags"))

orders_clean.show(truncate=False)


### Capstone Task 2 — Aggregation & Analysis [Write from scratch]

Using `orders_clean` from Task 1:

1. **Monthly revenue summary**: group by month (extract from `order_date`), compute total revenue (`sum`) and order count (`count`), sorted by month ascending
2. **Category analysis**: for each `category`, compute total revenue, average order value, and number of unique customers. Sort by total revenue descending.

Show both results.


In [ ]:
# YOUR CODE HERE — Capstone Task 2

# Monthly revenue summary:
monthly_revenue = orders_clean \
    .groupBy(F.month("order_date").alias("month")) \
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("order_id").alias("order_count")
    ) \
    .orderBy("month")

monthly_revenue.show()

# Category analysis:
category_analysis = orders_clean \
    .groupBy("category") \
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("avg_order_value"),
        F.countDistinct("customer_name").alias("unique_customers")
    ) \
    .orderBy("total_revenue", ascending=False)

category_analysis.show()


### Capstone Task 3 — Window Functions [Write from scratch]

Using `orders_clean` from Task 1:

1. For each `category`, rank orders by `amount` (highest first) — call it `rank_in_category`
2. Filter to show only the **top 2** orders per category
3. Show: `category`, `customer_name`, `amount`, `rank_in_category`


In [ ]:
# YOUR CODE HERE — Capstone Task 3
windowSpec = Window.partitionBy("category").orderBy(F.col("amount").desc())

result = orders_clean \
    .withColumn("rank_in_category", F.rank().over(windowSpec)) \
    .filter(F.col("rank_in_category") <= 2) \
    .select("category", "customer_name", "amount", "rank_in_category") \
    .orderBy("category", "rank_in_category")

result.show()


---
## Well Done!

Check your score, review any exercises you found difficult, and compare with the answer key once you are ready.

| Section | Your score |
|---------|-----------|
| 3 — DataFrame Ops (20 pts) | &nbsp; |
| 4 — Built-in & Aggregates (16 pts) | &nbsp; |
| 5 — String Functions (16 pts) | &nbsp; |
| 6 — Date & Time (18 pts) | &nbsp; |
| 7 — Array Functions (12 pts) | &nbsp; |
| 8 — Capstone (18 pts) | &nbsp; |
| **Total (100 pts)** | &nbsp; |


---
# Lab 1

## Section 1 — Practice Exercises

In [ ]:
students = spark.createDataFrame([
    (101, "Ali",     "CS",  88),
    (102, "Mona",    "CS",  95),
    (103, "Youssef", "IT",  72),
    (104, "Nour",    "IT",  81),
    (105, "Sara",    "AI",  91),
    (106, "Omar",    "AI",  85)
], ["student_id", "name", "major", "score"])
students.show()

### Exercise 1 — Filter AI students with score > 90

In [ ]:
result = students.filter((F.col("major") == "AI") & (F.col("score") > 90))
result.show()

### Exercise 2 — Add `result` column (Pass / Fail)

In [ ]:
result = students.withColumn(
    "result",
    F.when(F.col("score") >= 80, "Pass").otherwise("Fail")
)
result.show()

### Exercise 3 — Average score per major (sorted highest → lowest)

In [ ]:
result = (students
    .groupBy("major")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
result.show()

### Exercise 4 — Count distinct majors

In [ ]:
result = students.select(F.countDistinct("major").alias("distinct_majors"))
result.show()

### Exercise 5 — Rename `major` → `department`, drop `student_id`

In [ ]:
result = students.withColumnRenamed("major", "department").drop("student_id")
result.show()

## Section 2 — Window Function Exercises

### Exercise 1 — Running total of scores ordered by student_id

In [ ]:
w = Window.orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = students.withColumn("running_total", F.sum("score").over(w))
result.show()

### Exercise 2 — above_major_avg / below_major_avg

In [ ]:
w_major = Window.partitionBy("major")
result = (students
    .withColumn("major_avg", F.avg("score").over(w_major))
    .withColumn("vs_major_avg",
        F.when(F.col("score") >= F.col("major_avg"), "above_major_avg")
         .otherwise("below_major_avg"))
    .drop("major_avg")
)
result.show()

### Exercise 3 — Top 2 students per major

In [ ]:
w_rank = Window.partitionBy("major").orderBy(F.col("score").desc())
result = (students
    .withColumn("rank", F.rank().over(w_rank))
    .filter(F.col("rank") <= 2)
)
result.show()

## Section 3 — String Function Exercises

In [ ]:
contacts = spark.createDataFrame([
    ("  ali123@example.com ",),
    ("sara_99@gmail.com",),
    ("mona.test@yahoo.com",),
    (" ahmed2025@outlook.com ",)
], ["email"])
contacts.show(truncate=False)

### Exercise 1 — Extract username and email provider

In [ ]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("username", F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

### Exercise 2 — Domain name only (gmail, yahoo, outlook …)

In [ ]:
result = (contacts
    .withColumn("email",  F.trim(F.lower(F.col("email"))))
    .withColumn("domain", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

### Exercise 3 — Clean email (trim + lowercase)

In [ ]:
result = contacts.withColumn("email", F.trim(F.lower(F.col("email"))))
result.show(truncate=False)

### Exercise 4 — Count emails per provider

In [ ]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
    .groupBy("provider")
    .agg(F.count("*").alias("email_count"))
)
result.show()

### Exercise 5 — Remove digits from usernames

In [ ]:
result = (contacts
    .withColumn("email",             F.trim(F.lower(F.col("email"))))
    .withColumn("username_raw",      F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("username_no_digits",F.regexp_replace("username_raw", r'[0-9]', ""))
)
result.select("email", "username_raw", "username_no_digits").show(truncate=False)

## Section 4 — Date & Time Exercises

In [ ]:
events = spark.createDataFrame([
    (1, "2025-01-15"),
    (2, "2025-03-22"),
    (3, "2025-07-04"),
    (4, "2025-10-19"),
    (5, "2025-12-31")
], ["event_id", "event_date"])
events.show()

### Exercise 1 — Convert event_date to DateType

In [ ]:
events = events.withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))
events.printSchema()
events.show()

### Exercise 2 — Extract year, month, day

In [ ]:
result = (events
    .withColumn("year",  F.year("event_date"))
    .withColumn("month", F.month("event_date"))
    .withColumn("day",   F.dayofmonth("event_date"))
)
result.show()

### Exercise 3 — Quarter of each event

In [ ]:
result = events.withColumn(
    "quarter",
    F.when(F.month("event_date").between(1, 3),  "Q1")
     .when(F.month("event_date").between(4, 6),  "Q2")
     .when(F.month("event_date").between(7, 9),  "Q3")
     .otherwise("Q4")
)
result.show()

### Exercise 4 — Days remaining until end of event's year

In [ ]:
result = events.withColumn(
    "days_until_year_end",
    F.datediff(
        F.to_date(F.concat(F.year("event_date").cast("string"), F.lit("-12-31")), "yyyy-MM-dd"),
        F.col("event_date")
    )
)
result.show()

### Exercise 5 — Events in the second half of the year (July–December)

In [ ]:
result = events.filter(F.month("event_date") >= 7)
result.show()

## Section 5 — Array Function Exercises

In [ ]:
courses = spark.createDataFrame([
    ("Python",  ["Programming", "Beginner", "Data"]),
    ("Spark",   ["Big Data", "Programming"]),
    ("SQL",     ["Database", "Data"]),
    ("Airflow", ["Workflow", "Automation", "Data"]),
    ("Linux",   ["OS", "Administration"])
], ["course_name", "topics"])
courses.show(truncate=False)

### Exercise 1 — Courses with more than 2 topics

In [ ]:
result = courses.filter(F.size("topics") > 2)
result.show(truncate=False)

### Exercise 2 — has_data_topic column (True/False)

In [ ]:
result = courses.withColumn("has_data_topic", F.array_contains("topics", "Data"))
result.show(truncate=False)

### Exercise 3 — Count courses per topic using explode()

In [ ]:
result = (courses
    .withColumn("topic", F.explode("topics"))
    .groupBy("topic")
    .agg(F.count("course_name").alias("course_count"))
    .orderBy("course_count", ascending=False)
)
result.show()

### Exercise 4 — Courses containing the topic "Programming"

In [ ]:
result = courses.filter(F.array_contains("topics", "Programming"))
result.show(truncate=False)

### Exercise 5 — Number of topics per course

In [ ]:
result = courses.withColumn("topic_count", F.size("topics"))
result.show(truncate=False)

## Capstone — Online Learning Platform Analytics

In [ ]:
students_cap = spark.createDataFrame([
    (1, "Ali"), (2, "Mona"), (3, "Sara"), (4, "Omar"), (5, "Nour")
], ["student_id", "student_name"])

enrollments = spark.createDataFrame([
    (1, "Python", 88), (1, "SQL", 92),
    (2, "Python", 95), (2, "Spark", 90),
    (3, "SQL",    78), (3, "Spark", 85),
    (4, "Python", 70),
    (5, "Spark",  98)
], ["student_id", "course", "score"])

print("students_cap:"); students_cap.show()
print("enrollments:");  enrollments.show()

### Part 1 — Data Preparation
#### Task 1 — Join students with enrollments

In [ ]:
joined = students_cap.join(enrollments, on="student_id", how="inner")
joined.show()

#### Task 2 — Schema and data quality

In [ ]:
joined.printSchema()
print("Row count:", joined.count())
print("Null counts:")
joined.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in joined.columns]).show()

#### Task 3 — Remove duplicate records

In [ ]:
joined = joined.dropDuplicates()
print("Row count after dedup:", joined.count())

### Part 2 — Aggregations
#### Task 4 — Average score per student

In [ ]:
avg_per_student = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_student.show()

#### Task 5 — Average score per course

In [ ]:
avg_per_course = (joined
    .groupBy("course")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_course.show()

#### Task 6 — Highest score per course

In [ ]:
highest_per_course = (joined
    .groupBy("course")
    .agg(F.max("score").alias("highest_score"))
)
highest_per_course.show()

#### Task 7 — Student with the highest average score overall

In [ ]:
result = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
    .limit(1)
)
result.show()

### Part 3 — Window Functions
#### Task 8 — Rank students within each course

In [ ]:
w_course = Window.partitionBy("course").orderBy(F.col("score").desc())
ranked = joined.withColumn("course_rank", F.rank().over(w_course))
ranked.show()

#### Task 9 — Running total of scores within each course

In [ ]:
w_running = Window.partitionBy("course").orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = joined.withColumn("running_total", F.sum("score").over(w_running))
result.show()

#### Task 10 — Difference between student score and course average

In [ ]:
w_avg = Window.partitionBy("course")
result = (joined
    .withColumn("course_avg", F.avg("score").over(w_avg))
    .withColumn("diff_from_avg", F.col("score") - F.col("course_avg"))
)
result.show()

### Part 4 — Business Rules
#### Task 11 — Performance category

In [ ]:
with_perf = joined.withColumn(
    "performance",
    F.when(F.col("score") >= 90, "Excellent")
     .when(F.col("score") >= 80, "Good")
     .otherwise("Needs Improvement")
)
with_perf.show()

#### Task 12 — Count students per performance category

In [ ]:
result = (with_perf
    .groupBy("performance")
    .agg(F.count("student_id").alias("student_count"))
    .orderBy("student_count", ascending=False)
)
result.show()

### Part 5 — Final Report

In [ ]:
w_final = Window.partitionBy("course").orderBy(F.col("score").desc())

final_report = (joined
    .withColumn("performance",
        F.when(F.col("score") >= 90, "Excellent")
         .when(F.col("score") >= 80, "Good")
         .otherwise("Needs Improvement"))
    .withColumn("course_rank", F.rank().over(w_final))
    .select("student_name", "course", "score", "course_rank", "performance")
    .orderBy("course", "course_rank")
)
final_report.show()

---
# Lab 2

## Section 1 — Practice Exercises

In [ ]:
employees = spark.createDataFrame([
    (1, "Anna",  "Engineering", 95000),
    (2, "Ben",   "Marketing",   72000),
    (3, "Clara", "Engineering", 88000),
    (4, "Dan",   "HR",          65000),
    (5, "Eva",   "Marketing",   79000),
    (6, "Frank", "Engineering", 91000),
], ["id", "name", "dept", "salary"])
employees.show()

### Exercise 1 — Engineering employees with salary > 90,000

In [ ]:
result = employees.filter((F.col("dept") == "Engineering") & (F.col("salary") > 90000))
result.show()

### Exercise 2 — salary_grade column (High / Standard)

In [ ]:
result = employees.withColumn(
    "salary_grade",
    F.when(F.col("salary") >= 80000, "High").otherwise("Standard")
)
result.show()

### Exercise 3 — Average salary per department (sorted descending)

In [ ]:
result = (employees
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_salary"))
    .orderBy("avg_salary", ascending=False)
)
result.show()

### Exercise 4 — Count distinct departments

In [ ]:
result = employees.select(F.countDistinct("dept").alias("distinct_depts"))
result.show()

### Exercise 5 — Rename `dept` → `department`, drop `id`

In [ ]:
result = employees.withColumnRenamed("dept", "department").drop("id")
result.show()

## Section 2 — Window Function Exercises

### Exercise 1 — Running total of salary ordered by id

In [ ]:
w = Window.orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = employees.withColumn("running_salary_total", F.sum("salary").over(w))
result.show()

### Exercise 2 — above_avg / below_avg per department

In [ ]:
w_dept = Window.partitionBy("dept")
result = (employees
    .withColumn("dept_avg", F.avg("salary").over(w_dept))
    .withColumn("vs_dept_avg",
        F.when(F.col("salary") >= F.col("dept_avg"), "above_avg")
         .otherwise("below_avg"))
    .drop("dept_avg")
)
result.show()

### Exercise 3 — Top 3 Engineering employees by salary (window function)

In [ ]:
w_eng = Window.partitionBy("dept").orderBy(F.col("salary").desc())
result = (employees
    .withColumn("rank", F.rank().over(w_eng))
    .filter((F.col("dept") == "Engineering") & (F.col("rank") <= 3))
)
result.show()

## Section 3 — String Function Exercises

In [ ]:
logs = spark.createDataFrame([
    ("2025-06-01 08:15:32 ERROR disk full",),
    ("2025-06-01 09:00:00 INFO  service started",),
    ("2025-06-02 14:45:11 WARN  memory low",),
    ("2025-06-03 23:59:59 ERROR connection timeout",),
], ["log_entry"])

emails_df = spark.createDataFrame([
    ("alice@gmail.com",),
    ("bob.smith@company.org",),
    ("clara99@yahoo.com",),
], ["email"])

names_df = spark.createDataFrame([
    ("  john doe 1st  ",),
    (" JANE  SMITH2  ",),
    ("  ali HASSAN  ",),
], ["name"])

### Exercise 1 — Extract date and log level from log entries

In [ ]:
result = (logs
    .withColumn("log_date",  F.regexp_extract("log_entry", r'^(\d{4}-\d{2}-\d{2})', 1))
    .withColumn("log_level", F.trim(F.regexp_extract("log_entry", r'\d{2}:\d{2}:\d{2}\s+(\S+)', 1)))
)
result.show(truncate=False)

### Exercise 2 — Extract username and domain from email

In [ ]:
result = (emails_df
    .withColumn("username", F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("domain",   F.regexp_extract("email", r'@(.+)$', 1))
)
result.show(truncate=False)

### Exercise 3 — Clean name: trim, proper case, remove digits

In [ ]:
result = (names_df
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("name", F.initcap(F.col("name")))
    .withColumn("name", F.regexp_replace("name", r'[0-9]', ""))
    .withColumn("name", F.trim(F.col("name")))
)
result.show(truncate=False)

## Section 4 — Date & Time Exercises

In [ ]:
orders = spark.createDataFrame([
    (1, "2025-01-05"),
    (2, "2025-04-12"),
    (3, "2025-06-21"),
    (4, "2025-09-07"),
    (5, "2025-11-29"),
], ["order_id", "order_date"])
orders = orders.withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
orders.show()

### Exercise 1 — Order age in days, quarter, last day of month

In [ ]:
result = (orders
    .withColumn("age_days",   F.datediff(F.current_date(), "order_date"))
    .withColumn("quarter",
        F.when(F.month("order_date").between(1, 3),  "Q1")
         .when(F.month("order_date").between(4, 6),  "Q2")
         .when(F.month("order_date").between(7, 9),  "Q3")
         .otherwise("Q4"))
    .withColumn("last_day_of_month", F.last_day("order_date"))
)
result.show()

### Exercise 2 — Orders placed on a weekend (Saturday or Sunday)

In [ ]:
# dayofweek: 1=Sunday, 7=Saturday
result = orders.filter(F.dayofweek("order_date").isin(1, 7))
result.show()

## Section 5 — Array Function Exercises

In [ ]:
products = spark.createDataFrame([
    (1, "Laptop",  ["electronics", "computers", "work"]),
    (2, "T-Shirt", ["clothing", "fashion"]),
    (3, "Phone",   ["electronics", "mobile"]),
    (4, "Headset", ["electronics", "audio", "computers"]),
    (5, "Jacket",  ["clothing", "fashion", "sale"]),
], ["id", "product", "tags"])

promo_products = spark.createDataFrame([
    (1, ["sale", "featured"]),
    (2, ["new",  "sale"]),
    (3, ["featured"]),
    (4, ["sale", "electronics"]),
    (5, ["clearance"]),
], ["id", "promo_tags"])

### Exercise 1 — Products with more than 2 tags

In [ ]:
result = products.filter(F.size("tags") > 2)
result.show(truncate=False)

### Exercise 2 — has_sale_tag column from promo_tags

In [ ]:
result = promo_products.withColumn("has_sale_tag", F.array_contains("promo_tags", "sale"))
result.show(truncate=False)

### Exercise 3 — Count products per tag using explode() + groupBy()

In [ ]:
result = (products
    .withColumn("tag", F.explode("tags"))
    .groupBy("tag")
    .agg(F.count("id").alias("product_count"))
    .orderBy("product_count", ascending=False)
)
result.show()